In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import scipy.optimize as sco
import matplotlib.pyplot as plt

# 1. Configurazione Parametri
tickers = ['SPY', 'VGK', 'EWJ', 'EEM', 'AAXJ']
start_date = '2015-01-01'
end_date = '2024-12-31'
risk_free_rate = 0.04 # 4% Tasso privo di rischio approssimativo (es. US Treasury)

# 2. Acquisizione Dati
data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']
returns = data.pct_change().dropna()

# Calcolo rendimenti medi annualizzati e matrice di covarianza
# 252 giorni di trading
mean_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

# 3. Funzioni di Performance del Portafoglio
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.sum(mean_returns * weights)
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std

# Funzione da minimizzare per Sharpe Ratio (negativo perché scipy fa solo minimizzazione)
def neg_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
    p_ret, p_std = portfolio_performance(weights, mean_returns, cov_matrix)
    return - (p_ret - risk_free_rate) / p_std

# Funzione da minimizzare per Volatilità
def portfolio_volatility(weights, mean_returns, cov_matrix):
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]

# 4. Ottimizzazione
num_assets = len(tickers)
args = (mean_returns, cov_matrix)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}) # Somma pesi = 1
bounds = tuple((0.0, 1.0) for asset in range(num_assets)) # Long only

# A. Max Sharpe Portfolio
init_guess = num_assets * [1. / num_assets,]
result_sharpe = sco.minimize(neg_sharpe_ratio, init_guess, args=(mean_returns, cov_matrix, risk_free_rate),
                             method='SLSQP', bounds=bounds, constraints=constraints)
weights_sharpe = result_sharpe.x

# B. Min Volatility Portfolio
result_minvol = sco.minimize(portfolio_volatility, init_guess, args=args,
                             method='SLSQP', bounds=bounds, constraints=constraints)
weights_minvol = result_minvol.x

# 5. Output dei Risultati
def print_portfolio(name, weights, ret, std):
    print(f"--- {name} ---")
    print(f"Rendimento Atteso: {ret:.2%}")
    print(f"Volatilità (Rischio): {std:.2%}")
    print(f"Sharpe Ratio: {(ret - risk_free_rate)/std:.2f}")
    print("Allocazione:")
    for t, w in zip(tickers, weights):
        print(f"  {t}: {w:.2%}")
    print("\n")

ret_s, std_s = portfolio_performance(weights_sharpe, mean_returns, cov_matrix)
print_portfolio("Max Sharpe Ratio Portfolio", weights_sharpe, ret_s, std_s)

ret_mv, std_mv = portfolio_performance(weights_minvol, mean_returns, cov_matrix)
print_portfolio("Minimum Variance Portfolio", weights_minvol, ret_mv, std_mv)

# 6. Visualizzazione Frontiera Efficiente (Simulazione Monte Carlo per visualizzazione)
num_portfolios = 5000
results = np.zeros((3, num_portfolios))
for i in range(num_portfolios):
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    p_ret, p_std = portfolio_performance(weights, mean_returns, cov_matrix)
    results[0,i] = p_std
    results[1,i] = p_ret
    results[2,i] = (p_ret - risk_free_rate) / p_std # Sharpe

plt.figure(figsize=(10, 6))
plt.scatter(results[0,:], results[1,:], c=results[2,:], cmap='viridis', marker='o', s=10, alpha=0.3)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(std_s, ret_s, marker='*', color='r', s=500, label='Max Sharpe')
plt.scatter(std_mv, ret_mv, marker='*', color='b', s=500, label='Min Volatility')
plt.title('Frontiera Efficiente con ETF Nazionali')
plt.xlabel('Volatilità (Deviazione Standard Annualizzata)')
plt.ylabel('Rendimento Atteso Annualizzato')
plt.legend()
plt.grid(True)
plt.show()

C:\Users\doze9\AppData\Local\Temp\ipykernel_25500\2425157409.py:14: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']
[*********************100%***********************]  5 of 5 completed


KeyError: 'Adj Close'